# PyTorch Computer Vision

## 0. Computer Vision Libraries in PyTorch

* `torchvision` - base domain library for PyTorch
* `torchvision.datasets` - get datasets and data loading functions for CV
* `torchvision.models` - get pretrained CV models
* `torchvision.transforms` - functions for manipulating vision data (images) to be suitable for use with an ML model
* `torch.utils.data.Data` - Base Dataset class for PyTorch
* `torch.utils.data.DataLoader` - Creates a Python iterable over a dataset.

In [ ]:
import torch
import torch.nn as nn

import torchvision
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor

import matplotlib.pyplot as plt

print(torch.__version__)
print(torchvision.__version__)

## 1. Getting a dataset

Fashion-MNIST dataset

In [ ]:
# Setup train data
train_data = datasets.FashionMNIST(root="data",
                                   train=True,
                                   download=True,
                                   transform=ToTensor())
test_data = datasets.FashionMNIST(root='data',
                                  train=False,
                                  download=True,
                                  transform=ToTensor())

In [ ]:
len(train_data), len(test_data)

In [ ]:
# see the first training example
image, label = train_data[0]
image, label

In [ ]:
class_names = train_data.classes
class_names

In [ ]:
class_to_idx = train_data.class_to_idx
class_to_idx

In [ ]:
train_data.targets

### 1.1 Check image and label shapes

In [ ]:
# check shape of our image
print(f"Image shape: {image.shape} -> [C, H, W]")
print(f"Image label: {class_names[label]}")

### 1.2 Visualize data

Note: Matplotlib does not use (C, H, W). It expects H, W (for grayscale) or expects color channel to be last for colored images.

In [ ]:
import matplotlib.pyplot as plt
image, label = train_data[0]
print(f"Image shape: {image.shape}")
plt.imshow(image.squeeze())
plt.title(label)

In [ ]:
plt.imshow(image.squeeze(), cmap="gray")
plt.title(class_names[label])
plt.axis(False)

In [ ]:
# Plot more images
# torch.manual_seed(42)
fig = plt.figure(figsize=(9, 9))
rows, cols = 4, 4
for i in range(1, rows*cols + 1):
  random_idx = torch.randint(0, len(train_data), size=[1]).item()
  img, lbl = train_data[random_idx]
  fig.add_subplot(rows, cols, i)
  plt.imshow(img.squeeze(), cmap="gray")
  plt.title(class_names[lbl])
  plt.axis(False)

In [ ]:
train_data, test_data

## 2. Prepare DataLoader
Currently, data is form of PyTorch Datasets (above).

DataLoader turnsour dataset into Python iterable. More specifically we want to turn data into batches (or mini-batches)

Why do we do this?

1. Computing hardware may not be able to look (store in memory) at 60000 images at once. So we break it down to 32 images at a time (batch_size of 32).
2. It gives NN more chances to update its gradients per epoch.

In [ ]:
from torch.utils.data import DataLoader

# Setup batch size hyperparamter
BATCH_SIZE = 32

train_data_loader = DataLoader(dataset=train_data,
                               batch_size=BATCH_SIZE,
                               shuffle=True)
test_data_loader = DataLoader(dataset=test_data,
                              batch_size=BATCH_SIZE,
                              shuffle=False)
train_data_loader, test_data_loader

In [ ]:
print(f"DataLoaders: {train_data_loader, test_data_loader}")
print(f"Length of train_data_loader: {len(train_data_loader)} batches of {BATCH_SIZE}")
print(f"Length of test_data_loader: {len(test_data_loader)} batches of {BATCH_SIZE}")

In [ ]:
# Check what's in the train dataloader
train_features_batch, train_labels_batch = next(iter(train_data_loader))
train_features_batch.shape, train_labels_batch.shape

In [ ]:
# # Show a sample
# torch.manual_seed(42)
random_idx = torch.randint(0, len(train_features_batch), size=[1]).item()
img, lbl = train_features_batch[random_idx], train_labels_batch[random_idx]
plt.imshow(img.squeeze(), cmap="gray")
plt.title(class_names[lbl])
plt.axis(False)
print(f"Image size: {img.shape}")
print(f"Label: {lbl}, label.shape: {lbl.shape}")

## 3. Model 0: Build a baseline model


In [ ]:
# Create a flatten layer
flatten_model = nn.Flatten()

# Get a single sample
x = train_features_batch[0]

# flatten the sample
output = flatten_model(x) # perform forward pass

print(f"Shape before flatten: {x.shape} -> [C, H, W]")
print(f"Shape after flatten: {output.shape} -> [C, H*W]")

In [ ]:
from torch import nn

class FashionMNISTModelV0(nn.Module):
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int = 10):
    super().__init__()
    self.layer_stack = nn.Sequential(
        nn.Flatten(), # flatten external dimensions
        nn.Linear(in_features=input_shape, out_features=hidden_units),
        nn.Linear(in_features=hidden_units, out_features=output_shape)
    )

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.layer_stack(x)

In [ ]:
torch.manual_seed(42)

model_0 = FashionMNISTModelV0(
    input_shape=784, # this is 28 * 28
    hidden_units=10, # hidden layer
    output_shape=len(class_names)
).to("cpu")
model_0

In [ ]:
dummy_x = torch.rand([1, 1, 28, 28])
model_0(dummy_x)

### 3.1 Setup loss, optimizer and evaluation metrics

In [ ]:
import requests
from pathlib import Path

# Download helper functions from learn pytorch repo
if Path("helper_functions.py").is_file():
  print("helper_functions.py already exists, skipping download...")
else:
  print("Downloading helper_functions.py")
  request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/refs/heads/main/helper_functions.py")
  with open("helper_functions.py", "wb") as f:
    f.write(request.content)

In [ ]:
# Import accuracy metric
from helper_functions import accuracy_fn

# Setup loss and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_0.parameters(),
                            lr=0.1)

### 3.2 Creating a function to time our experiments

things to track:
1. Model's performance (loss and accuracy values)
2. How fast it runs (latency)

In [ ]:
from timeit import default_timer as timer

def print_train_time(start: float,
                     end: float,
                     device: torch.device = None):
  """Prints difference between start and end time."""
  total_time = end - start
  print(f"Train time on {device}: {total_time:.3f} seconds")
  return total_time

In [ ]:
start_time = timer()
# some code
end_time = timer()
print_train_time(start_time, end_time, device="cpu")

### 3.3 Create a train loop (on batches of data)

1. Loop through epochs
2. Loop through training batches, perform training steps, calculate train loss *per batch*
3. Loop through test batches, perform testing steps calculate test loss *per batch*
4. Print out what's happening
5. Time it all

In [ ]:
len(train_data_loader.dataset)

In [ ]:
# Import tqdm for progress bar
from tqdm.auto import tqdm # recognize what compute environment we are using, give us best type of progress bar for it. For eg. Jupyter notebook vs python scripts

# Set the seed and start the timer
torch.manual_seed(42)
train_time_start_on_cpu = timer()

# Set the number of epochs
epochs = 3

for epoch in tqdm(range(epochs)): # wrap iterator with tqdm
  print(f"Epoch: {epoch}\n-----")

  # Training
  model_0.train()
  train_loss = 0.0
  train_acc = 0.0
  # loop through training batches
  for batch_idx, (images, labels) in enumerate(train_data_loader):
    y_logits = model_0(images)
    y_pred = torch.argmax(torch.softmax(y_logits, dim=1), dim=1)

    loss = loss_fn(y_logits, labels)
    acc = accuracy_fn(labels, y_pred)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_loss += loss.item() # accumulate loss per batch
    train_acc += acc

    if batch_idx % 400 == 0:
      print(f"Looked at {batch_idx * len(images)} / {len(train_data_loader.dataset)} samples.")

  # Total train loss
  train_loss /= len(train_data_loader)
  train_acc /= len(train_data_loader)

  # Testing
  model_0.eval()
  test_loss = 0.0
  test_acc = 0.0
  with torch.inference_mode():
    # loop through test batches
    for images, labels in test_data_loader:
      test_logits = model_0(images)
      test_pred = torch.argmax(torch.softmax(test_logits, dim=1), dim=1)

      loss = loss_fn(test_logits, labels)
      acc = accuracy_fn(labels, test_pred)

      test_loss += loss.item() # accumulate loss per batch
      test_acc += acc

    # Total train loss
    test_loss /= len(test_data_loader)
    test_acc /= len(test_data_loader)

  print(f"\nTrain Loss: {train_loss:.4f} | Train acc: {train_acc} | Test Loss: {test_loss:.4f} | Test acc: {test_acc}")

# Calculate time
train_time_end_on_cpu = timer()
total_train_time_model_0 = print_train_time(train_time_start_on_cpu,
                                            train_time_end_on_cpu,
                                            device=str(next(model_0.parameters()).device))



## 4. Make Predictions and get model_0 results

In [ ]:
torch.manual_seed(42)
def eval_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               accuracy_fn,
               device: torch.device):
  """Return a dict of model results on data_loader"""
  loss, acc = 0.0, 0.0
  model.to(device)
  model.eval()

  with torch.inference_mode():
    for X, y in tqdm(data_loader):
      # Put data on target device
      X = X.to(device)
      y = y.to(device)
      # Forward pass
      y_logits = model(X)

      # Accumulate loss and acc values per batch
      loss += loss_fn(y_logits, y)
      acc += accuracy_fn(y, torch.argmax(torch.softmax(y_logits, dim=1), dim=1))

    # Scale loss and acc to find avg loss/acc per batch
    loss /= len(data_loader)
    acc /= len(data_loader)

  return {"model_name": model.__class__.__name__,
          "model_loss": loss.item(),
          "model_acc": acc}

# calculate model_0 results on test dataset

model_0_results = eval_model(model_0, test_data_loader, loss_fn, accuracy_fn, torch.device("cpu"))
model_0_results

## 5. Setup device agnostic code

In [ ]:
!nvidia-smi

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 6. Model 1: Build model with non-linearity

In [ ]:
from torch.nn.modules.activation import ReLU
class FashionMNISTModelV1(nn.Module):
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int = 10):
    super().__init__()
    self.layer_stack = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=input_shape, out_features=hidden_units),
        nn.ReLU(),
        nn.Linear(in_features=hidden_units, out_features=output_shape),
        nn.ReLU()
    )

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.layer_stack(x)

dummy_model_1 = FashionMNISTModelV1(28*28, 10, 10)
dummy_model_1

In [ ]:
torch.manual_seed(42)
model_1 = FashionMNISTModelV1(28*28, 10, 10).to(device)
next(model_1.parameters()).device

### 6.1 Loss, optimizer, Evaluation metrics

In [ ]:
from helper_functions import accuracy_fn
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_1.parameters(), lr=0.1)

### 6.2 functioning train/test loops

In [ ]:
def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               accuracy_fn,
               device: torch.device = device):
  train_loss = 0.0
  train_acc = 0.0
  # # Put model in train mode
  # model.train()
  # loop through training batches
  for batch_idx, (images, labels) in enumerate(data_loader):
    # Put on target device
    images = images.to(device)
    labels = labels.to(device)

    y_logits = model(images)
    y_pred = torch.argmax(torch.softmax(y_logits, dim=1), dim=1)

    loss = loss_fn(y_logits, labels)
    acc = accuracy_fn(labels, y_pred)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_loss += loss.item() # accumulate loss per batch
    train_acc += acc

  # Total train loss
  train_loss /= len(data_loader)
  train_acc /= len(data_loader)
  print(f"Train Loss: {train_loss:.5f} | Train acc: {train_acc:.2f}%")

In [ ]:
def test_step(model: torch.nn.Module,
              data_loader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              accuracy_fn,
              device: torch.device = device):
  test_loss = 0.0
  test_acc = 0.0
  # put model in eval mode
  model.eval()
  with torch.inference_mode():
    # loop through test batches
    for images, labels in data_loader:
      # Put on target device
      images = images.to(device)
      labels = labels.to(device)

      test_logits = model(images)
      test_pred = torch.argmax(torch.softmax(test_logits, dim=1), dim=1)

      loss = loss_fn(test_logits, labels)
      acc = accuracy_fn(labels, test_pred)

      test_loss += loss.item() # accumulate loss per batch
      test_acc += acc

    # Total train loss
    test_loss /= len(data_loader)
    test_acc /= len(data_loader)
    print(f"Test Loss: {test_loss:.5f} | Test acc: {test_acc:.2f}%")


In [ ]:
torch.manual_seed(42)

# Measure time
from timeit import default_timer as timer
train_time_start_on_gpu = timer()

# Set epochs
epochs = 3

for epoch in tqdm(range(epochs)):
  print(f"Epoch: {epoch}\n")
  train_step(model_1, train_data_loader, loss_fn, optimizer, accuracy_fn, device)
  test_step(model_1, test_data_loader, loss_fn, accuracy_fn, device)

train_time_end_on_gpu = timer()
total_train_time_model_1 = print_train_time(train_time_start_on_gpu,
                                            train_time_end_on_gpu,
                                            device)

In [ ]:
device

**Note**: Sometimes, we observe dependeing on our data / hardware, model runs faster on CPU than GPU.

Why is this?

1. It could be that the overhead of copying data / model to and from GPU outweighs the compute benefits offered by GPU
2. The hardware we are using has a better CPU in terms of compute capability than the GPU.

In [ ]:
# Get model_1 results dict
model_1_results = eval_model(model_1, test_data_loader, loss_fn, accuracy_fn, device)
model_1_results

In [ ]:
model_0_results

## 7. Model 2: Build a CNN

https://poloclub.github.io/cnn-explainer/

In [ ]:
class FashionMNISTModelV2(nn.Module):
  """Model architecture which replicates TinyVGG from CNN explainer website"""

  def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
    super().__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=input_shape,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1), # hyperparamters for Conv2d
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2) # stride=Default is kernel_size
    )

    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1), # hyperparamters for Conv2d
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2) # stride=Default is kernel_size
        )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_units*7*7,
                  out_features=output_shape) # num_classes
    )

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    x = self.conv_block_1(x)
    x = self.conv_block_2(x)
    x = self.classifier(x) # do not need to flatten x as the classifier does it already.
    return x


In [ ]:
torch.manual_seed(42)

model_2 = FashionMNISTModelV2(input_shape=1, # (1, 28, 28)
                               hidden_units=10,
                               output_shape=len(class_names)).to(device)
model_2

### 7.1 Stepping through `nn.Conv2d`

In [ ]:
torch.manual_seed(42)

# create batch of images
images = torch.randn(size=(32, 3, 64, 64))
test_image = images[0]
print(f"Images batch shape: {images.shape}")
print(f"Test image shape: {test_image.shape}")
print(f"Test image:\n {test_image}")

In [ ]:
# Create a single conv2d layer
conv_layer = nn.Conv2d(in_channels=3, out_channels=10, kernel_size=3, stride=1, padding=1)
# pass data through the layer
conv_out = conv_layer(test_image)
conv_out.shape

### 7.2 Stepping through `nn.MaxPool2d`

In [ ]:
test_image.shape

In [ ]:
max_pool_layer = nn.MaxPool2d(kernel_size=2)
max_pool_out = max_pool_layer(conv_out)
max_pool_out.shape

### 7.3 Setup Loss and Optimizer for model_2

In [ ]:
from helper_functions import accuracy_fn

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_2.parameters(), lr=0.1)

### 7.4 Train and test model_2 using our train and test functions

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Measure time
from timeit import default_timer as timer
train_time_start_model_2 = timer()

epochs = 3
for epoch in tqdm(range(epochs)):
  print(f"Epoch: {epoch}\n-------")
  train_step(model_2, train_data_loader, loss_fn, optimizer, accuracy_fn, device)
  test_step(model_2, test_data_loader, loss_fn, accuracy_fn, device)

train_time_end_model_2 = timer()

total_train_time_model_2 = print_train_time(train_time_start_model_2,
                                            train_time_end_model_2,
                                            device)


In [ ]:
# Get model_2 results
model_2_results = eval_model(model_2, test_data_loader, loss_fn, accuracy_fn, device)
model_2_results

In [ ]:
model_0_results

## 8. Compare model results and training time

In [ ]:
import pandas as pd

compare_results = pd.DataFrame([model_0_results, model_1_results, model_2_results])
compare_results

In [ ]:
compare_results['training_time'] = [total_train_time_model_0,
                                    total_train_time_model_1,
                                    total_train_time_model_2]
compare_results

In [ ]:
# Visualize our model results
compare_results.set_index("model_name")["model_acc"].plot(kind="barh")
plt.xlabel("accuracy (%) ")
plt.ylabel("model")

## 9. Make and evaluate random predictions with the best model

In [ ]:
def make_predictions(model: torch.nn.Module,
                     data: list,
                     device: torch.device = device):
  pred_probs = []
  model.to(device)
  model.eval()
  with torch.inference_mode():
    for sample in data:
      # Prepare the sample
      sample = torch.unsqueeze(sample, dim=0).to(device)
      sample = sample.to(device)
      # forward pass
      pred_logits = model(sample)

      pred_prob = torch.softmax(pred_logits.squeeze(), dim=0)

      pred_probs.append(pred_prob.cpu())

  return torch.stack(pred_probs)


In [ ]:
img, lbl = test_data[0]
img.shape, lbl

In [ ]:
import random

# random.seed(42)

test_samples = []
test_labels = []

for sample, label in random.sample(list(test_data), k=9):
  test_samples.append(sample)
  test_labels.append(label)

# View the first sample shape
test_samples[0].shape

In [ ]:
plt.imshow(test_samples[0].squeeze(), cmap="gray")

In [ ]:
# Make predictions
pred_probs = make_predictions(model_2, test_samples, device)

# View first 2 prediction probs
pred_probs[:2].shape

In [ ]:
# Convert pred_prob to labels
pred_classes = pred_probs.argmax(dim=1)
pred_classes

In [ ]:
test_labels

In [ ]:
# Plot Predictions
plt.figure(figsize=(9, 9))
nrows = 3
ncols = 3
for i, sample in enumerate(test_samples):
  # Create subplot
  plt.subplot(nrows, ncols, i+1)
  # Plot the target image
  plt.imshow(sample.squeeze(), cmap="gray")
  # Find prediction (in text form)
  pred_label = class_names[pred_classes[i]]
  # truth label
  truth_label = class_names[test_labels[i]]
  # title for plot
  title_text = f"Pred: {pred_label} | Truth: {truth_label}"

  if pred_label == truth_label:
    plt.title(title_text, fontsize=10, c='g')
  else:
    plt.title(title_text, fontsize=10, c='r')

  plt.axis(False)

## 10. Making a confusion matrix for further prediction evaluation

1. Make predictions with our trained model on test data.
2. Make confusion matrix using `torchmetrics`
3. Plot the confusion matrix using `mlxtend.plotting.plot_confusion_matrix()`

In [ ]:
# Make predictions
y_preds = []
model_2.to(device)
model_2.eval()
with torch.inference_mode():
  for X, y in tqdm(test_data_loader, desc="Making predictions..."):
    X, y = X.to(device), y.to(device)
    y_logits = model_2(X)
    y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)
    y_preds.append(y_pred.cpu())

# print(y_preds)
y_pred_tensor = torch.cat(y_preds)
y_pred_tensor

In [ ]:
len(y_pred_tensor)

In [ ]:
import sys

# Install required packages
try:
  import torchmetrics, mlxtend
  print(f"mlxtend version: {mlxtend.__version__}")
  assert int(mlxtend.__version__.split(".")[1]) >= 19, "mlxtend should be 0.19.0 or higher"
except:
  print("Reinstalling torchmetrics and mlxtend due to import error...")
  !pip install --upgrade torchmetrics mlxtend

# Verify installation after potential reinstallation (this part will only run if no initial error or after restart)
try:
  import torchmetrics, mlxtend
  print(f"mlxtend version: {mlxtend.__version__}")
  assert int(mlxtend.__version__.split(".")[1]) >= 19, "mlxtend should be 0.19.0 or higher"
except Exception as e:
  print(f"An unexpected error occurred after installation: {e}")

In [ ]:
mlxtend.__version__

In [ ]:
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix

# Setup Conf Matrix instance and compare preds with target
conf_mat = ConfusionMatrix(task='multiclass', num_classes=len(class_names))
conf_mat_tensor = conf_mat(preds=y_pred_tensor, target=test_data.targets)
conf_mat_tensor

In [ ]:
# Plot our confusion matrix
fig, ax = plot_confusion_matrix(conf_mat=conf_mat_tensor.numpy(),
                                class_names=class_names,
                                figsize=(10, 7))

## 11. Save and Load model (best performing)

In [ ]:
from pathlib import Path

MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True,
                 exist_ok=True)
MODEL_NAME = "03_pytorch_computer_vision_model_2.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

# Save the model state dict
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model_2.state_dict(), f=MODEL_SAVE_PATH)

In [ ]:
# Create a new instance
torch.manual_seed(42)

loaded_model_2 = FashionMNISTModelV2(input_shape=1, hidden_units=10, output_shape=len(class_names))
# Load the saved model dict
loaded_model_2.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

# Send the model to target device
loaded_model_2.to(device)

In [ ]:
model_2_results

In [ ]:
# Evaluate loaded model
torch.manual_seed(42)

loaded_model_2_results = eval_model(model=loaded_model_2,
                                    data_loader=test_data_loader,
                                    loss_fn=loss_fn,
                                    accuracy_fn=accuracy_fn,
                                    device=device)
loaded_model_2_results

In [ ]:
# Check if model results are close to each other
torch.isclose(torch.tensor(model_2_results["model_loss"]),
              torch.tensor(loaded_model_2_results["model_loss"]),
              atol=1e-02)

In [ ]:
torch.allclose(torch.tensor(model_2_results["model_loss"]),
               torch.tensor(loaded_model_2_results["model_loss"]),
               atol=1e-02)

## 12. Exercises

In [ ]:
# Check for GPU
!nvidia-smi

In [ ]:
# Import torch
import torch

# Exercises require PyTorch > 1.10.0
print(torch.__version__)

# TODO: Setup device agnostic code
device = "cuda" if torch.cuda.is_available else "cpu"
device

1. What are 3 areas in industry where computer vision is currently being used?

Robotics, Medicine, Agriculture

2. Search "what is overfitting in machine learning" and write down a sentence about what you find.

Overfitting in ML happens when a model learns the training data too well. Instead of learning general, real world patterns, model tends to learn specific patterns, noise and outliers.

3. Search "ways to prevent overfitting in machine learning", write down 3 of the things you find and a sentence about each.

* Data augmentation (diverse, large set): Model can learn general rules instead of memorizing specific examples.

* Regularization: Add a regularization term to the loss function. When the reduction in Training Loss is bigger than the Penalty increase, the model keeps the weight. If a weight only helps a tiny bit, the Penalty cost outweighs the benefit, then model shrinks or removes that weight to lower the Total Loss.

* Early Stopping: Monitor model's performance on a separate validation dataset during training. If the validation loss stops improving or begins to increase, use early stopping to halt the training process before the model begins memorizing the training data.

In [ ]:
# 5. Load the torchvision.datasets.MNIST() train and test datasets.
from torch.utils.data import DataLoader


train_data = torchvision.datasets.MNIST(root="/data",
                                        train=True,
                                        transform=torchvision.transforms.ToTensor(),
                                        download=True)
test_data = torchvision.datasets.MNIST(root="/data",
                                       train=False,
                                       transform=torchvision.transforms.ToTensor(),
                                       download=True)

len(train_data), len(test_data)

In [ ]:
len(train_data.classes)

In [ ]:
# 6. Visualize at least 5 different samples of the MNIST training dataset.
for i in range(5):
  img, lbl = train_data[i]
  plt.subplot(1, 5, i + 1)
  plt.imshow(img.squeeze(), cmap="gray")
  plt.title(str(lbl))
  plt.axis(False)

In [ ]:
# 7. Turn the MNIST train and test datasets into dataloaders using torch.utils.data.DataLoader, set the batch_size=32
train_data_loader = DataLoader(dataset=train_data,
                               batch_size=32,
                               shuffle=True)
test_data_loader = DataLoader(dataset=test_data,
                              batch_size=32,
                              shuffle=False)
len(train_data_loader), len(test_data_loader)

In [ ]:
# 8. Recreate model_2 used in notebook 03 (the same model from the CNN Explainer website, also known as TinyVGG) capable of fitting on the MNIST dataset.
class TinyVGG(nn.Module):
  def __init__(self, in_channels: int, hidden_units: int, out_channels: int):
    super().__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=in_channels,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2)
    )
    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2)
    )
    self.classifier = nn.Linear(in_features=hidden_units*7*7,
                                out_features=out_channels)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.classifier(self.conv_block_2(self.conv_block_1(x)))


In [ ]:
model_3 = TinyVGG(1, 10, len(train_data.classes))
model_3